## 1. Imports

In [1]:
import os
import warnings

import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")

## 2. Paths

In [2]:
PROJECT_PATH = "/content/drive/MyDrive/FinSight_AI"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data"
)

PROCESSED_PATH = os.path.join(
    DATA_PATH,
    "processed"
)

PROFILE_PATH = os.path.join(
    PROCESSED_PATH,
    "company_profiles"
)

TIMELINE_PATH = os.path.join(
    PROCESSED_PATH,
    "company_timeline"
)

ANALYTICS_PATH = os.path.join(
    PROCESSED_PATH,
    "company_analytics"
)

os.makedirs(
    ANALYTICS_PATH,
    exist_ok=True
)

## 3. Load Datasets

In [4]:
company_profiles = pd.read_parquet(

    os.path.join(

        PROFILE_PATH,

        "company_profiles.parquet"

    )

)

timeline_df = pd.read_parquet(

    os.path.join(

        TIMELINE_PATH,

        "company_event_timeline.parquet"

    )

)

print("Company Profiles")
print(company_profiles.shape)

print()

print("Company Timeline")
print(timeline_df.shape)

Company Profiles
(6619, 60)

Company Timeline
(3215296, 26)


## 4. Normalize Risk Score

In [5]:
# Normalized Risk Score
scaler = MinMaxScaler(

    feature_range=(0,100)

)

company_profiles["normalized_risk_score"] = scaler.fit_transform(

    company_profiles[

        [

            "risk_score"

        ]

    ]

)

company_profiles["normalized_risk_score"] = (

    company_profiles["normalized_risk_score"]

    .round(2)

)

company_profiles[[

    "ticker",

    "risk_score",

    "normalized_risk_score"

]].head()

,ticker,risk_score,normalized_risk_score
0,A,-467,24.23
1,AA,-85,35.59
2,AAC,-22,37.47
3,AADR,0,38.12
4,AAL,65,40.05


## 5. Risk Level

In [6]:
def risk_level(score):

    if score >= 80:

        return "Very High"

    elif score >= 60:

        return "High"

    elif score >= 40:

        return "Medium"

    elif score >= 20:

        return "Low"

    else:

        return "Very Low"


company_profiles["risk_level"] = (

    company_profiles["normalized_risk_score"]

    .apply(risk_level)

)

company_profiles.head()

,ticker,total_news,first_news,latest_news,unique_publishers,avg_headline_length,avg_word_count,avg_confidence,median_confidence,min_confidence,...,Regulatory Approval,Revenue Guidance,Share Buyback,Stock Split,Technical Analysis,Trading Halt,risk_rank,intelligence_rank,normalized_risk_score,risk_level
0,A,2328,2009-04-29 00:00:00+00:00,2020-06-05 14:30:54+00:00,126,67.441581,10.307990,0.846600,0.78160,0.250,...,33,26,9,0,52,2,688,106,24.23,Low
1,AA,2908,2009-08-10 00:00:00+00:00,2020-06-09 14:52:15+00:00,205,63.069120,10.222146,0.842717,0.76785,0.333,...,7,54,4,12,118,9,349,7,35.59,Low
2,AAC,366,2010-03-25 00:00:00+00:00,2019-11-12 00:00:00+00:00,20,67.387978,10.592896,0.861256,0.85700,0.455,...,1,1,3,1,12,5,286,3035,37.47,Low
3,AADR,32,2012-04-19 00:00:00+00:00,2020-04-30 00:00:00+00:00,5,48.781250,8.000000,0.768309,0.74000,0.500,...,0,1,0,0,2,0,264,5665,38.12,Low
4,AAL,494,2011-05-16 00:00:00+00:00,2020-06-10 15:21:01+00:00,21,109.107287,17.010121,0.825818,0.75930,0.333,...,6,16,5,2,19,14,200,2922,40.05,Medium


## 6. Event Diversity Score

In [7]:
# Event Diversity
event_columns = [

    c

    for c in company_profiles.columns

    if c in timeline_df["final_event"].unique()

]

company_profiles["event_diversity"] = (

    company_profiles[event_columns]

    .gt(0)

    .sum(axis=1)

)

company_profiles[[

    "ticker",

    "event_diversity"

]].head()

,ticker,event_diversity
0,A,29
1,AA,30
2,AAC,25
3,AADR,11
4,AAL,25


## 7. Market Stability Score

In [8]:
# Market Stability Score
company_profiles["market_stability_score"] = (

    company_profiles["Bullish_pct"]

    +

    company_profiles["Neutral_pct"]

    -

    company_profiles["Bearish_pct"]

).clip(

    lower=0,

    upper=100

)

company_profiles[

    [

        "ticker",

        "Bullish_pct",

        "Bearish_pct",

        "Neutral_pct",

        "market_stability_score"

    ]

].head()

,ticker,Bullish_pct,Bearish_pct,Neutral_pct,market_stability_score
0,A,13.27,1.98,84.32,95.61
1,AA,6.29,2.85,90.51,93.95
2,AAC,4.92,6.01,87.70,86.61
3,AADR,3.12,0.00,96.88,100.00
4,AAL,12.15,7.09,78.95,84.01


## 8. News Momentum Score

In [9]:
# News Momentum
latest_date = timeline_df["published_date"].max()

last_30 = latest_date - pd.Timedelta(days=30)

last_90 = latest_date - pd.Timedelta(days=90)

last_year = latest_date - pd.Timedelta(days=365)

momentum = (

    timeline_df

    .groupby("ticker")

    .agg(

        total_news=("news_id","count"),

        news_last_30=(

            "published_date",

            lambda x: (x >= last_30).sum()

        ),

        news_last_90=(

            "published_date",

            lambda x: (x >= last_90).sum()

        ),

        news_last_year=(

            "published_date",

            lambda x: (x >= last_year).sum()

        )

    )

    .reset_index()

)

momentum.head()

,ticker,total_news,news_last_30,news_last_90,news_last_year
0,A,2328,23,71,298
1,AA,2908,14,74,295
2,AAC,366,0,0,50
3,AADR,32,0,2,4
4,AAL,494,46,150,238


## 9. Activity Score

In [10]:
# Activity Score
activity = company_profiles[

    [

        "ticker",

        "total_news",

        "unique_publishers",

        "event_diversity"

    ]

].merge(

    momentum,

    on="ticker"

)

activity_features = [

    "total_news_x",

    "unique_publishers",

    "event_diversity",

    "news_last_30",

    "news_last_90",

    "news_last_year"

]

activity_scaler = MinMaxScaler()

activity_scaled = activity_scaler.fit_transform(

    activity[activity_features]

)

activity["activity_score"] = (

    activity_scaled.mean(axis=1)

    * 100

)

activity["activity_score"] = (

    activity["activity_score"]

    .round(2)

)

activity.head()

,ticker,total_news_x,unique_publishers,event_diversity,total_news_y,news_last_30,news_last_90,news_last_year,activity_score
0,A,2328,126,29,2328,23,71,298,38.71
1,AA,2908,205,30,2908,14,74,295,45.73
2,AAC,366,20,25,366,0,0,50,16.71
3,AADR,32,5,11,32,0,2,4,6.21
4,AAL,494,21,25,494,46,150,238,26.96


## 10. Composite Intelligence Score

In [11]:
# Composite Intelligence Score
analytics = company_profiles.merge(

    activity[

        [

            "ticker",

            "activity_score"

        ]

    ],

    on="ticker"

)

analytics["composite_score"] = (

      0.30 * analytics["activity_score"]

    + 0.25 * analytics["market_stability_score"]

    + 0.20 * analytics["avg_confidence"] * 100

    + 0.15 * analytics["event_diversity"] / analytics["event_diversity"].max() * 100

    + 0.10 * (100 - analytics["normalized_risk_score"])

)

analytics["composite_score"] = (

    analytics["composite_score"]

    .round(2)

)

analytics[[

    "ticker",

    "activity_score",

    "market_stability_score",

    "normalized_risk_score",

    "composite_score"

]].head()

,ticker,activity_score,market_stability_score,normalized_risk_score,composite_score
0,A,38.71,95.61,24.23,74.52
1,AA,45.73,93.95,35.59,75.50
2,AAC,16.71,86.61,37.47,62.64
3,AADR,6.21,100.00,38.12,53.92
4,AAL,26.96,84.01,40.05,64.10


## 21. Company Rankings

In [12]:
# Company Rankings
analytics["overall_rank"] = (

    analytics["composite_score"]

    .rank(

        ascending=False,

        method="dense"

    )

    .astype(int)

)

analytics = analytics.sort_values(

    "overall_rank"

)

analytics.head(20)

,ticker,total_news,first_news,latest_news,unique_publishers,avg_headline_length,avg_word_count,avg_confidence,median_confidence,min_confidence,...,Trading Halt,risk_rank,intelligence_rank,normalized_risk_score,risk_level,event_diversity,market_stability_score,activity_score,composite_score,overall_rank
2523,GILD,4938,2009-09-11 00:00:00+00:00,2020-06-10 17:44:51+00:00,169,75.379911,11.686513,0.820607,0.75350,0.286,...,13,769,6,10.29,Very Low,30,96.41,83.09,89.41,1
2746,HD,4415,2009-08-10 00:00:00+00:00,2020-06-10 12:14:08+00:00,232,64.542922,10.525481,0.829553,0.75990,0.333,...,14,557,1,29.29,Low,30,96.08,74.00,84.88,2
3778,MDT,4421,2009-08-07 00:00:00+00:00,2020-06-11 14:22:31+00:00,134,68.356254,10.373445,0.851852,0.78050,0.333,...,4,777,20,0.00,Very Low,30,97.71,57.43,83.69,3
5442,SLV,3968,2009-08-17 00:00:00+00:00,2020-06-03 00:00:00+00:00,119,50.743196,8.517137,0.849060,0.78610,0.250,...,4,166,43,41.21,Medium,28,92.37,72.31,81.65,4
3644,LOW,3212,2009-08-13 00:00:00+00:00,2020-06-11 14:34:01+00:00,189,62.575965,10.007161,0.843545,0.76450,0.357,...,7,700,10,23.55,Low,29,96.09,59.47,80.88,5
3486,KR,5061,2009-09-15 00:00:00+00:00,2020-06-11 16:04:32+00:00,168,58.813278,9.536060,0.827025,0.75640,0.333,...,12,735,5,20.04,Low,30,95.11,58.47,80.86,6
3608,LLY,3498,2009-08-31 00:00:00+00:00,2020-06-11 09:40:14+00:00,149,77.799314,12.140938,0.838436,0.76685,0.333,...,7,759,31,13.83,Very Low,28,94.92,58.27,80.60,7
2159,FDX,4358,2009-09-13 00:00:00+00:00,2020-06-09 14:41:45+00:00,209,64.468564,10.303809,0.831489,0.76180,0.167,...,25,266,2,38.06,Low,30,93.58,64.03,80.43,8
505,AXP,3741,2009-08-07 00:00:00+00:00,2020-06-10 14:22:11+00:00,188,65.549318,10.324779,0.828504,0.76040,0.333,...,14,652,11,26.05,Low,30,94.87,57.90,80.05,9
4994,REGN,3597,2010-03-21 00:00:00+00:00,2020-06-05 13:29:07+00:00,114,71.700028,10.671949,0.830609,0.75750,0.333,...,26,726,82,21.17,Low,30,96.03,54.60,79.88,10


## 12. Company Feature Vectors

In [13]:
# Company Feature Vectors
EVENT_COLUMNS = sorted(

    timeline_df["final_event"].unique()

)

MARKET_COLUMNS = [

    "Bullish_pct",

    "Bearish_pct",

    "Neutral_pct"

]

FEATURE_COLUMNS = (

    EVENT_COLUMNS

    +

    MARKET_COLUMNS

    +

    [

        "avg_confidence",

        "activity_score",

        "event_diversity",

        "market_stability_score",

        "normalized_risk_score"

    ]

)

company_vectors = analytics[

    [

        "ticker"

    ]

    +

    FEATURE_COLUMNS

].copy()

company_vectors.head()

,ticker,Analyst Rating,Bankruptcy,Commodity,Credit Rating,Cryptocurrency,Cybersecurity,Dividend,ESG,Earnings,...,Technical Analysis,Trading Halt,Bullish_pct,Bearish_pct,Neutral_pct,avg_confidence,activity_score,event_diversity,market_stability_score,normalized_risk_score
2523,GILD,425,1,121,46,10,8,96,22,552,...,124,13,10.63,1.62,87.40,0.820607,83.09,30,96.41,10.29
2746,HD,355,3,118,57,7,51,208,25,712,...,172,14,12.91,1.70,84.87,0.829553,74.00,30,96.08,29.29
3778,MDT,395,1,39,46,1,9,256,13,588,...,58,4,17.55,1.02,81.18,0.851852,57.43,30,97.71,0.00
5442,SLV,12,4,2717,19,31,8,4,7,67,...,180,4,6.00,3.68,90.05,0.849060,72.31,28,92.37,41.21
3644,LOW,419,6,43,28,4,9,188,9,548,...,83,7,14.98,1.74,82.85,0.843545,59.47,29,96.09,23.55


## 13. Normalize Feature Vectors

In [14]:
# Normalize Feature Vectors
vector_scaler = MinMaxScaler()

X = vector_scaler.fit_transform(

    company_vectors[

        FEATURE_COLUMNS

    ]

)

print(X.shape)

(6619, 38)


## 14. Company Similarity Matrix

In [15]:
# Company Similarity Matrix
similarity_matrix = cosine_similarity(

    X

)

print(similarity_matrix.shape)

(6619, 6619)


## 15. Build Top-10 Similar Companies

In [17]:
# Top Similar Companies
TOP_K = 10

similarity_records = []

tickers = company_vectors["ticker"].values

for i in range(len(tickers)):

    similarities = similarity_matrix[i]

    top_indices = np.argsort(similarities)[::-1][1:TOP_K+1]

    for idx in top_indices:

        similarity_records.append({

            "ticker": tickers[i],

            "similar_company": tickers[idx],

            "similarity_score": round(

                similarities[idx],

                4

            )

        })

company_similarity = pd.DataFrame(

    similarity_records

)

company_similarity.head(10)

,ticker,similar_company,similarity_score
0,GILD,LLY,0.9591
1,GILD,BMY,0.9445
2,GILD,REGN,0.9389
3,GILD,MRK,0.9385
4,GILD,MDT,0.9324
5,GILD,AZN,0.9178
6,GILD,BSX,0.9159
7,GILD,KO,0.9149
8,GILD,VRTX,0.9119
9,GILD,NFLX,0.9083


## 16. Company Clustering

In [18]:
# Company Clustering
N_CLUSTERS = 12

kmeans = KMeans(

    n_clusters=N_CLUSTERS,

    random_state=42,

    n_init=20

)

analytics["cluster"] = kmeans.fit_predict(

    X

)

analytics.head()

,ticker,total_news,first_news,latest_news,unique_publishers,avg_headline_length,avg_word_count,avg_confidence,median_confidence,min_confidence,...,risk_rank,intelligence_rank,normalized_risk_score,risk_level,event_diversity,market_stability_score,activity_score,composite_score,overall_rank,cluster
2523,GILD,4938,2009-09-11 00:00:00+00:00,2020-06-10 17:44:51+00:00,169,75.379911,11.686513,0.820607,0.7535,0.286,...,769,6,10.29,Very Low,30,96.41,83.09,89.41,1,3
2746,HD,4415,2009-08-10 00:00:00+00:00,2020-06-10 12:14:08+00:00,232,64.542922,10.525481,0.829553,0.7599,0.333,...,557,1,29.29,Low,30,96.08,74.00,84.88,2,3
3778,MDT,4421,2009-08-07 00:00:00+00:00,2020-06-11 14:22:31+00:00,134,68.356254,10.373445,0.851852,0.7805,0.333,...,777,20,0.00,Very Low,30,97.71,57.43,83.69,3,3
5442,SLV,3968,2009-08-17 00:00:00+00:00,2020-06-03 00:00:00+00:00,119,50.743196,8.517137,0.849060,0.7861,0.250,...,166,43,41.21,Medium,28,92.37,72.31,81.65,4,11
3644,LOW,3212,2009-08-13 00:00:00+00:00,2020-06-11 14:34:01+00:00,189,62.575965,10.007161,0.843545,0.7645,0.357,...,700,10,23.55,Low,29,96.09,59.47,80.88,5,3


## 17. Cluster Statistics

In [19]:
# Cluster Statistics
cluster_summary = (

    analytics

    .groupby("cluster")

    .agg(

        companies=("ticker","count"),

        avg_activity=("activity_score","mean"),

        avg_risk=("normalized_risk_score","mean"),

        avg_score=("composite_score","mean")

    )

    .round(2)

)

display(cluster_summary)

,companies,avg_activity,avg_risk,avg_score
cluster,,,,
0,1020,21.77,34.20,65.63
1,994,4.78,38.19,52.15
2,944,16.01,37.62,62.20
3,148,43.74,35.22,74.38
4,430,31.79,33.72,70.17
5,289,6.52,35.96,55.78
6,446,11.21,36.61,56.40
7,420,3.91,37.83,53.19
8,1033,9.43,37.51,57.41


## 18. Automatic Cluster Labels

In [20]:
# Cluster Categories
def assign_cluster_category(row):

    if row["avg_risk"] >= 70:

        return "High Risk"

    elif row["avg_activity"] >= 70:

        return "High Activity"

    elif row["avg_score"] >= 75:

        return "High Intelligence"

    elif row["avg_risk"] <= 20:

        return "Stable"

    else:

        return "Balanced"

cluster_summary["category"] = (

    cluster_summary

    .apply(

        assign_cluster_category,

        axis=1

    )

)

display(cluster_summary)

,companies,avg_activity,avg_risk,avg_score,category
cluster,,,,,
0,1020,21.77,34.20,65.63,Balanced
1,994,4.78,38.19,52.15,Balanced
2,944,16.01,37.62,62.20,Balanced
3,148,43.74,35.22,74.38,Balanced
4,430,31.79,33.72,70.17,Balanced
5,289,6.52,35.96,55.78,Balanced
6,446,11.21,36.61,56.40,Balanced
7,420,3.91,37.83,53.19,Balanced
8,1033,9.43,37.51,57.41,Balanced


## 19. Company Fingerprint

In [21]:
# Company Fingerprint
def company_fingerprint(row):

    tags = []

    if row["normalized_risk_score"] >= 70:
        tags.append("High Risk")

    elif row["normalized_risk_score"] <= 20:
        tags.append("Stable")

    if row["activity_score"] >= 70:
        tags.append("High Activity")

    if row["market_stability_score"] >= 75:
        tags.append("Bullish")

    elif row["market_stability_score"] <= 40:
        tags.append("Bearish")

    if row["event_diversity"] >= 15:
        tags.append("Diversified")

    return ", ".join(tags)


analytics["company_fingerprint"] = analytics.apply(

    company_fingerprint,

    axis=1

)

analytics[

    [

        "ticker",

        "company_fingerprint"

    ]

].head()

,ticker,company_fingerprint
2523,GILD,"Stable, High Activity, Bullish, Diversified"
2746,HD,"High Activity, Bullish, Diversified"
3778,MDT,"Stable, Bullish, Diversified"
5442,SLV,"High Activity, Bullish, Diversified"
3644,LOW,"Bullish, Diversified"


## 20. Risk Explanation

In [22]:
# Risk Explainability
RISK_EVENTS = [

    "Fraud",

    "Bankruptcy",

    "Cybersecurity",

    "Litigation",

    "Layoffs"

]

def explain_risk(row):

    explanation = []

    for event in RISK_EVENTS:

        if event in row.index and row[event] > 0:

            explanation.append(

                f"{event}: {int(row[event])}"

            )

    if len(explanation) == 0:

        return "No major risk events"

    return " | ".join(explanation)


analytics["risk_explanation"] = analytics.apply(

    explain_risk,

    axis=1

)

analytics[

    [

        "ticker",

        "normalized_risk_score",

        "risk_explanation"

    ]

].head()

,ticker,normalized_risk_score,risk_explanation
2523,GILD,10.29,Fraud: 3 | Bankruptcy: 1 | Cybersecurity: 8 | ...
2746,HD,29.29,Fraud: 6 | Bankruptcy: 3 | Cybersecurity: 51 |...
3778,MDT,0.00,Fraud: 2 | Bankruptcy: 1 | Cybersecurity: 9 | ...
5442,SLV,41.21,Fraud: 1 | Bankruptcy: 4 | Cybersecurity: 8 | ...
3644,LOW,23.55,Bankruptcy: 6 | Cybersecurity: 9 | Litigation:...


## 21. Save Analytics Dataset

In [23]:
# Save Analytics
analytics_output = os.path.join(

    ANALYTICS_PATH,

    "company_analytics.parquet"

)

analytics.to_parquet(

    analytics_output,

    index=False

)

print("Company Analytics Saved")
print(analytics_output)

Company Analytics Saved
/content/drive/MyDrive/FinSight_AI/data/processed/company_analytics/company_analytics.parquet


## 22. Save Company Similarity

In [24]:
# Save Similarity
similarity_output = os.path.join(

    ANALYTICS_PATH,

    "company_similarity.parquet"

)

company_similarity.to_parquet(

    similarity_output,

    index=False

)

print("Company Similarity Saved")
print(similarity_output)

Company Similarity Saved
/content/drive/MyDrive/FinSight_AI/data/processed/company_analytics/company_similarity.parquet


## 23. Save Company Clusters

In [25]:
# Save Clusters
cluster_output = os.path.join(

    ANALYTICS_PATH,

    "company_clusters.parquet"

)

cluster_summary.reset_index().to_parquet(

    cluster_output,

    index=False

)

print("Company Clusters Saved")
print(cluster_output)

Company Clusters Saved
/content/drive/MyDrive/FinSight_AI/data/processed/company_analytics/company_clusters.parquet


## 24. Save Feature Vectors

In [26]:
# Save Company Feature Vectors
vector_output = os.path.join(

    ANALYTICS_PATH,

    "company_event_vectors.parquet"

)

company_vectors.to_parquet(

    vector_output,

    index=False

)

print("Company Feature Vectors Saved")
print(vector_output)

Company Feature Vectors Saved
/content/drive/MyDrive/FinSight_AI/data/processed/company_analytics/company_event_vectors.parquet


## 25. Top 20 Companies

In [27]:
# Top Companies
display(

    analytics

    .sort_values(

        "overall_rank"

    )

    [

        [

            "ticker",

            "overall_rank",

            "composite_score",

            "activity_score",

            "normalized_risk_score",

            "market_stability_score",

            "company_fingerprint"

        ]

    ]

    .head(20)

)

,ticker,overall_rank,composite_score,activity_score,normalized_risk_score,market_stability_score,company_fingerprint
2523,GILD,1,89.41,83.09,10.29,96.41,"Stable, High Activity, Bullish, Diversified"
2746,HD,2,84.88,74.00,29.29,96.08,"High Activity, Bullish, Diversified"
3778,MDT,3,83.69,57.43,0.00,97.71,"Stable, Bullish, Diversified"
5442,SLV,4,81.65,72.31,41.21,92.37,"High Activity, Bullish, Diversified"
3644,LOW,5,80.88,59.47,23.55,96.09,"Bullish, Diversified"
3486,KR,6,80.86,58.47,20.04,95.11,"Bullish, Diversified"
3608,LLY,7,80.60,58.27,13.83,94.92,"Stable, Bullish, Diversified"
2159,FDX,8,80.43,64.03,38.06,93.58,"Bullish, Diversified"
505,AXP,9,80.05,57.90,26.05,94.87,"Bullish, Diversified"
4994,REGN,10,79.88,54.60,21.17,96.03,"Bullish, Diversified"


## 26. Example Similar Companies

In [28]:
# Similar Companies Example
example_company = "AAPL"

display(

    company_similarity[

        company_similarity["ticker"] == example_company

    ]

    .sort_values(

        "similarity_score",

        ascending=False

    )

)

,ticker,similar_company,similarity_score
2530,AAPL,FB,0.9967
2531,AAPL,AMZN,0.9950
2532,AAPL,LTM,0.9881
2533,AAPL,F,0.9879
2534,AAPL,CUK,0.9877
2535,AAPL,UNH,0.9872
2536,AAPL,TGT,0.9863
2537,AAPL,CHTR,0.9863
2538,AAPL,HPQ,0.9858
2539,AAPL,PSO,0.9857


## 27. Cluster Overview

In [29]:
display(cluster_summary)

,companies,avg_activity,avg_risk,avg_score,category
cluster,,,,,
0,1020,21.77,34.20,65.63,Balanced
1,994,4.78,38.19,52.15,Balanced
2,944,16.01,37.62,62.20,Balanced
3,148,43.74,35.22,74.38,Balanced
4,430,31.79,33.72,70.17,Balanced
5,289,6.52,35.96,55.78,Balanced
6,446,11.21,36.61,56.40,Balanced
7,420,3.91,37.83,53.19,Balanced
8,1033,9.43,37.51,57.41,Balanced


## 28.Analytics Summary

In [30]:
summary = pd.DataFrame({

    "Metric":[

        "Companies",

        "Similarity Records",

        "Clusters",

        "Average Composite Score",

        "Average Risk Score",

        "Highest Composite Score",

        "Highest Risk Score"

    ],

    "Value":[

        len(analytics),

        len(company_similarity),

        analytics["cluster"].nunique(),

        round(

            analytics["composite_score"].mean(),

            2

        ),

        round(

            analytics["normalized_risk_score"].mean(),

            2

        ),

        round(

            analytics["composite_score"].max(),

            2

        ),

        round(

            analytics["normalized_risk_score"].max(),

            2

        )

    ]

})

display(summary)

,Metric,Value
0,Companies,6619.00
1,Similarity Records,66190.00
2,Clusters,12.00
3,Average Composite Score,60.12
4,Average Risk Score,36.16
5,Highest Composite Score,89.41
6,Highest Risk Score,100.00
